# Hotel Analytics: Guest Value & Loyalty Segmentation (Python)

**Business question:** Who are our guests, and which groups are most valuable to the hotel?

**Audience:** Marketing Manager, Revenue Manager, General Manager

**Questions answered in this notebook**
1. What is the hotel's total revenue, and how is it split between lodging and other spending?
2. Which guests spend the most, and what share of revenue do they bring?
3. Which guests are loyal (stay often and recently)?
4. Which RFM segments appear, and how big and valuable is each?
5. Supporting question: who are the guests who never stayed?

**Notebook map:** 1. Setup → 2. Data preparation → 3. Analysis (Questions 1 to 5)

## 1. Setup: libraries and data

In [1]:
!pip install kagglehub -q

import kagglehub
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

path = kagglehub.dataset_download("nantonio/a-hotels-customers-dataset")
print("Path to dataset files:", path)

# See which files are in the folder
print(os.listdir(path))

100%|██████████| 15.6M/15.6M [00:00<00:00, 78.0MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/nantonio/a-hotels-customers-dataset/versions/2
['HotelCustomersDataset.xlsx']


In [2]:
df = pd.read_excel(os.path.join(path, "HotelCustomersDataset.xlsx"))

## 2. Data preparation

### 2.1 Check the data
Shape, data types, missing values, and the min/max of each column.

In [4]:
df.shape
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83590 entries, 0 to 83589
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    83590 non-null  int64  
 1   Nationality           83590 non-null  object 
 2   Age                   79811 non-null  float64
 3   DaysSinceCreation     83590 non-null  int64  
 4   NameHash              83590 non-null  object 
 5   DocIDHash             83590 non-null  object 
 6   AverageLeadTime       83590 non-null  int64  
 7   LodgingRevenue        83590 non-null  float64
 8   OtherRevenue          83590 non-null  float64
 9   BookingsCanceled      83590 non-null  int64  
 10  BookingsNoShowed      83590 non-null  int64  
 11  BookingsCheckedIn     83590 non-null  int64  
 12  PersonsNights         83590 non-null  int64  
 13  RoomNights            83590 non-null  int64  
 14  DaysSinceLastStay     83590 non-null  int64  
 15  DaysSinceFirstStay 

,0
ID,0
Nationality,0
Age,3779
DaysSinceCreation,0
NameHash,0
DocIDHash,0
AverageLeadTime,0
LodgingRevenue,0
OtherRevenue,0
BookingsCanceled,0


In [5]:
print(df.columns.tolist())
df.isna().sum()[df.isna().sum()>0]

['ID', 'Nationality', 'Age', 'DaysSinceCreation', 'NameHash', 'DocIDHash', 'AverageLeadTime', 'LodgingRevenue', 'OtherRevenue', 'BookingsCanceled', 'BookingsNoShowed', 'BookingsCheckedIn', 'PersonsNights', 'RoomNights', 'DaysSinceLastStay', 'DaysSinceFirstStay', 'DistributionChannel', 'MarketSegment', 'SRHighFloor', 'SRLowFloor', 'SRAccessibleRoom', 'SRMediumFloor', 'SRBathtub', 'SRShower', 'SRCrib', 'SRKingSizeBed', 'SRTwinBed', 'SRNearElevator', 'SRAwayFromElevator', 'SRNoAlcoholInMiniBar', 'SRQuietRoom']


,0
Age,3779


In [6]:
df.describe().T[['count','min','max','mean','50%']]

,count,min,max,mean,50%
ID,83590.0,1.0,83590.00,41795.500000,41795.5
Age,79811.0,-11.0,122.00,45.398028,46.0
DaysSinceCreation,83590.0,0.0,1095.00,453.640902,397.0
AverageLeadTime,83590.0,-1.0,588.00,66.196028,29.0
LodgingRevenue,83590.0,0.0,21781.00,298.802087,234.0
OtherRevenue,83590.0,0.0,7730.25,67.589132,38.5
BookingsCanceled,83590.0,0.0,9.00,0.002022,0.0
BookingsNoShowed,83590.0,0.0,3.00,0.000634,0.0
BookingsCheckedIn,83590.0,0.0,66.00,0.794617,1.0
PersonsNights,83590.0,0.0,116.00,4.649133,4.0


Counts of the problems found above. The last check counts **lodging** revenue only, so it is labeled that way.

In [ ]:
print("Age <18:",(df['Age']<18).sum())
print("Age>100:",(df['Age'] >100).sum())
print("Leadtime<0:",(df['AverageLeadTime']<0).sum())
print("DaysSinceLastStay<0:",(df['DaysSinceLastStay']<0).sum())
print("DaysSinceFirstStay<0:",(df['DaysSinceFirstStay']<0).sum())
print("Zero check-ins:",(df['BookingsCheckedIn']==0).sum())
print("Zero lodging revenue:",(df['LodgingRevenue']==0).sum())
print("Duplicate rows (ignoring ID):",df.drop(columns='ID').duplicated().sum())

### 2.2 Clean the data
**Rules:** ages below 18 or above 100 become NaN; negative lead times become NaN; `TotalRevenue` = lodging + other; guests with zero check-ins are flagged `NeverStayed`.

In [8]:
df_clean = df.copy()
df_clean.loc[(df_clean["Age"]<18)|(df_clean["Age"]>100),"Age"] = np.nan
df_clean.loc[(df_clean["AverageLeadTime"]<0),"AverageLeadTime"] = np.nan
df_clean['TotalRevenue'] = df_clean['LodgingRevenue']+df_clean['OtherRevenue']
df_clean['NeverStayed'] = df_clean['BookingsCheckedIn'] ==0

Check that revenue and check-ins line up, what the under-18 ages look like, and whether the duplicates matter.

In [9]:
#Do revenue and check-ins line up
print(pd.crosstab(df_clean['NeverStayed'],df_clean['TotalRevenue']==0))

#What do the under-18 ages look like
print(df.loc[df['Age']<18,'Age'].value_counts().sort_index().head(10))

#Are the duplicated mostly never-stayed guests?
dups = df_clean.drop(columns='ID').duplicated(keep=False)
print(df_clean.loc[dups,'NeverStayed'].value_counts())

TotalRevenue  False  True 
NeverStayed               
False         63385    285
True              0  19920
Age
-11.0      2
-10.0      4
-9.0       2
-7.0       3
-6.0       3
-1.0       3
 0.0      42
 1.0     118
 2.0     125
 3.0     125
Name: count, dtype: int64
NeverStayed
True     118
False     26
Name: count, dtype: int64


### 2.3 Build the RFM population and scores
RFM population = guests who stayed **and** have revenue. Frequency is scored by hand (most guests stayed once); Recency and Monetary use five equal-sized groups.

In [10]:
#Create the RFM group
rfm = df_clean[(~df_clean['NeverStayed'])&(df_clean['TotalRevenue']>0)].copy()
print(len(rfm))

63385


In [11]:
print(rfm['BookingsCheckedIn'].value_counts().sort_index().head(8))
print(rfm['BookingsCheckedIn'].value_counts(normalize=True).sort_index().head(8))

BookingsCheckedIn
1    61955
2     1130
3      129
4       56
5       20
6       20
7       14
8       10
Name: count, dtype: int64
BookingsCheckedIn
1    0.977439
2    0.017828
3    0.002035
4    0.000883
5    0.000316
6    0.000316
7    0.000221
8    0.000158
Name: proportion, dtype: float64


In [12]:
#Frequency: set by hand, because most guests have 1 stay
rfm['F_score'] = pd.cut(rfm['BookingsCheckedIn'],
                        bins=[0,1,2,np.inf],
                        labels=[1,2,3]).astype(int)
#Recency: fewer days since last stay = better, so labels run from 5 down to 1
rfm['R_score'] = pd.qcut(rfm['DaysSinceLastStay'],5,
                         labels=[5,4,3,2,1]).astype(int)
#Monetary: more revenue = better
rfm['M_score']=pd.qcut(rfm['TotalRevenue'],5,
                       labels=[1,2,3,4,5]).astype(int)
for c in ['R_score','F_score','M_score']:
  print(rfm[c].value_counts().sort_index())

R_score
1    12659
2    12689
3    12643
4    12665
5    12729
Name: count, dtype: int64
F_score
1    61955
2     1130
3      300
Name: count, dtype: int64
M_score
1    12715
2    12639
3    12706
4    12652
5    12673
Name: count, dtype: int64


## 3. Analysis

### Question 1: What is the hotel's total revenue, and how is it split between lodging and other spending?

In [14]:
#Question: What is the hotel’s total revenue, and how much comes from lodging vs. other spending?

Total_lodging_revenue=df_clean['LodgingRevenue'].sum()
print('Total_lodging_revenue:',Total_lodging_revenue)

Total_other_revenue=df_clean['OtherRevenue'].sum()
print('Total_other_revenue:',Total_other_revenue)

Total_revenue=df_clean['TotalRevenue'].sum()
print('Total_revenue:',Total_revenue)

Pct_lodging_revenue=Total_lodging_revenue*100/Total_revenue
print(f'Lodging share: {Pct_lodging_revenue:.1f}')

Pct_other_revenue=Total_other_revenue*100/Total_revenue
print(f'Other share: {Pct_other_revenue:.1f}')

Total_lodging_revenue: 24976866.46
Total_other_revenue: 5649775.53
Total_revenue: 30626641.99
Lodging share: 81.6
Other share: 18.4


### Question 2: Which guests spend the most, and what share of revenue do they bring?

#### 2a. How concentrated is revenue? (top 20% of guests)

In [ ]:
#What share of total revenue comes from the top 20% of guests?
cutoff = rfm['TotalRevenue'].quantile(0.8)
top20 = rfm[rfm['TotalRevenue']>=cutoff]

print(f'Cutoff revenue: {cutoff:.2f}')
print(f'Guests in top 20%: {len(top20)}')
print(f"Revenue from top 20%: {top20['TotalRevenue'].sum():.2f}")

top20_share = top20['TotalRevenue'].sum()/rfm['TotalRevenue'].sum()*100
print(f'Share of revenue: {top20_share:.1f}%')

#### 2b. Who are the top spenders? Nationality

In [17]:
shares = pd.DataFrame({
    'top20_%': top20['Nationality'].value_counts(normalize=True)*100,
    'all_%': rfm['Nationality'].value_counts(normalize=True)*100
})
shares['diff_pts']=shares['top20_%']-shares['all_%']
shares.sort_values('all_%',ascending=False).head(10).round(1)

,top20_%,all_%,diff_pts
Nationality,,,
FRA,16.6,14.8,1.9
PRT,8.6,14.7,-6.1
DEU,11.9,12.5,-0.5
GBR,10.4,10.4,0.0
ESP,5.8,6.2,-0.4
ITA,4.1,4.0,0.0
USA,3.2,3.8,-0.6
BEL,5.0,3.7,1.3
NLD,3.6,3.2,0.4


Average and median spend per guest for the five largest markets.

In [ ]:
five = rfm[rfm['Nationality'].isin(['FRA','PRT','DEU','GBR','ESP'])]
five.groupby('Nationality')['TotalRevenue'].agg(['mean','median','count']).round(1)

### Question 3: Which guests are loyal (stay often and recently)?

In [ ]:
#Question 3: which guests are loyal (stay often and recently)?
#1. Repeat guests = 2 or more stays (F_score of 2 or 3)
repeat = rfm[rfm['F_score']>=2]
print('Repeat guests: ',len(repeat))
print('Share of all RFM guests (%):',round(len(repeat)/len(rfm)*100,1))

#2. The ones who also stayed recently (R_score of 4 or 5)
recent_repeat = repeat[repeat['R_score']>=4]
print('Repeat AND recent:',len(recent_repeat))

#3. How much revenue do repeat guests bring
repeat_rev = repeat['TotalRevenue'].sum()
print('Repeat guests revenue: ',round(repeat_rev,2))
print('Share of all total revenue (%):',round(repeat_rev/rfm['TotalRevenue'].sum()*100,1))

### Question 4: Which RFM segments appear, and how big and valuable is each?
Rules are checked in order, and the first one that matches wins.

In [ ]:
conditions = [
    (rfm['F_score']>=2) & (rfm['M_score']>=4),
    (rfm['F_score']>=2),
    (rfm['M_score']>=4) & (rfm['R_score']>=4),
    (rfm['M_score']>=4),
    (rfm['M_score']<=2) & (rfm['R_score']<=2),
    (rfm['R_score']>=4),
]

names=['Loyal VIPs','Repeat guests','Recent big spenders',
       'Big spenders to win back','Lost low spenders','Recent guests']

rfm['Segment']=np.select(conditions,names,default='Mid-value, cooling off')

seg = rfm.groupby('Segment').agg(
    guests = ('ID','count'),
    revenue = ('TotalRevenue','sum'),
    avg_revenue = ('TotalRevenue','mean'),
    avg_days_since_last_stay =('DaysSinceLastStay','mean')
)

seg['guest_share_%']= (seg['guests']/seg['guests'].sum()*100).round(1)
seg['revenue_share_%']=(seg['revenue']/seg['revenue'].sum()*100).round(1)
seg.sort_values('revenue',ascending=False)

Chart: share of guests vs. share of revenue by segment.

In [ ]:
#1. Summarize each segment: number of guests and total revenue
rfm_seg = rfm.groupby('Segment').agg(
    guests = ('ID','count'),
    revenue = ('TotalRevenue','sum')
)

#2. Convert both to percentage of total
rfm_seg['guest_share'] = rfm_seg['guests']/rfm_seg['guests'].sum()*100
rfm_seg['revenue_share'] = rfm_seg['revenue']/rfm_seg['revenue'].sum()*100

#3. Sort so the biggest revenue segment ends up at the top of the chart
rfm_seg = rfm_seg.sort_values('revenue_share',ascending=True)

#4. Draw horizontal grouped bars
ax = rfm_seg[['guest_share','revenue_share']].plot(
    kind='barh',figsize=(9,5),color=['#9AA5B1','#4C78A8']
)

#5. Titles and legend
ax.set_title('Share of guests vs. share of revenue by RFM segment')
ax.set_xlabel('% of total')
ax.set_ylabel('')
ax.legend(['Share of guests','Share of revenue'])

#6. Write the value at the end of each bar
for container in ax.containers:
  ax.bar_label(container,fmt='%.1f',padding=3)

plt.tight_layout()
plt.savefig('rfm_segments.png',dpi=200,bbox_inches='tight')
plt.show()

### Supporting question 5: Who are the guests who never stayed?
Do they come from different booking channels and market segments than guests who stayed?

In [ ]:
df_clean['StayedStatus'] = np.where(df_clean['NeverStayed'],'NeverStayed','StayedGuest')
status_counts = df_clean['StayedStatus'].value_counts()
status_counts

Distribution channel (% of each group)

In [ ]:
pd.crosstab(df_clean['DistributionChannel'],df_clean['StayedStatus'],
            normalize='columns').mul(100).round(1)

Market segment (% of each group)

In [ ]:
pd.crosstab(df_clean['MarketSegment'],df_clean['StayedStatus'],
            normalize='columns').mul(100).round(1)